# 4. oturum: kredi ve karar araçlarının Python aynası

[![Colab'da aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alperozpinar/ftek503-2026/blob/main/haftalar/h03-excel-finans/h03-excel-finans.ipynb)

**Defterin hedefi:** `h03-excel-finans.xlsx` dosyasındaki hesapları (FV, PV, PMT, IPMT, NPV, IRR, RATE) Python'da `numpy-financial` kütüphanesiyle yeniden yapmak, iki sonucun aynı olduğunu `assert` ile göstermek ve uygulamanın G3 ile G4 görevlerini yazmak. NPV (net present value) net bugünkü değer, IRR (internal rate of return) iç verim oranıdır.

**Haftanın vakası**

| | |
|---|---|
| Soru | **TCMB'ye göre ihtiyaç kredisi faizi yıllık %63. İlandaki "aylık %3,89'dan başlayan faiz" bu ortalamanın neresinde; 100.000 TL için üç teklifin hangisinin maliyeti en düşük, hangisinin taksiti bütçeye sığar?** |
| Neden şimdi? | TCMB PPK toplantı özetine göre ihtiyaç kredisi (KMH hariç) faiz oranları 24 Temmuz ile biten haftaya göre 221 baz puan azalarak 4 Eylül 2026 ile biten haftada %63,0 olmuştur (Sayı 2026-42, 17.09.2026, 5. paragraf; https://www.tcmb.gov.tr/wps/wcm/connect/tr/tcmb+tr/main+menu/duyurular/basin/2026/duy2026-42, erişim 11.10.2026). TCMB bu oranı her kredinin efektif yıllık faiz oranından hesaplar. |
| Veri ve dağıtım | Veri dosyası yok. %63 elle, kaynak satırıyla girilir. Üç teklif, gelir ve bütçe eşiği varsayımsaldır; piyasa verisi değildir. Vergiler (KKDF, BSMV) modele dahil değildir. |
| Excel'de | Finans (f03), Hesap (FV, PV, PMT, amortisman, NPV, IRR), Karar (TCMB oranının hesabı, Goal Seek, Data Table, IF, AND, OR), Uygulama (G1, G2, G4) |
| Colab'da | Ayna: aynı hesaplar `numpy-financial` ile; G3 `taksit()`, `amortisman()` ve `maliyet()` fonksiyonları; G4 tutar × vade ızgarası |
| Hızlı şerit | TCMB metaverisindeki örneğin tarihli akışla (gün/365) yeniden bulunması; %63'e denk taksit (notsuz) |
| El kitabı | [f03: Paranın zaman değeri ve kredi](https://github.com/alperozpinar/ftek503-2026/blob/main/finans-temeli/f03-paranin-zaman-degeri-ve-kredi.md) |

**Çalışma biçimi (PRIMM):**
1. **Tahmin:** Hücre çalıştırılmadan önce sonuç tahmin edilir.
2. **Çalıştırma:** Hücreye tıklanır, Shift+Enter'a basılır.
3. **İnceleme:** Çıktı Excel'deki hücreyle karşılaştırılır.
4. **Değiştirme:** `DEĞİŞTİR` hücresinde bir sayı değiştirilir ve hücre yeniden çalıştırılır.
5. **Yazma:** `YAZ` hücresinde `None` yerine kısa bir formül yazılır.

**Etiketler** Excel dosyasındakiyle aynıdır: [G] Girdiler, [D] Dönem ve oran, [İ] İşaret, [H] Hesap, [S] Sağlama, [Y] Yorum.

**İlk adım:** Defter kişisel Drive'a kopyalanır: **Dosya > Drive'a kopya kaydet**. Hücreler yukarıdan aşağıya sırayla çalıştırılır; alttaki hücreler üsttekilerde tanımlanan adları kullanır.

**Python okuma ipuçları:**
- `#` ile başlayan satır açıklamadır; Python onu çalıştırmaz.
- `oran = 0.04` satırı, `oran` adlı değişkene 0,04 değerini atar. Excel'de bir hücreye sayı yazmaya karşılık gelir.
- Python'da ondalık ayırıcı **noktadır**. Excel'deki %4, Python'da `0.04` yazılır.
- `assert` bir koşulu sınar: koşul doğru değilse çalışma durur ve hata verilir. Excel'deki fark hücresinin Python karşılığıdır.

In [ ]:
# Sürüm hücresi: defterin hangi Python ve kütüphane sürümleriyle çalıştığını yazdırır.
import sys
import numpy as np
import pandas as pd
print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## Kurulum

`numpy-financial`, Excel'in finans fonksiyonlarının Python karşılıklarını içerir. Colab'da hazır gelmez; her oturumda bir kez kurulur.
- Sürüm sabitlenir (`==1.1.0`): herkes aynı sürümle aynı sonucu alır; uyumsuz bir ortamda kurulum sessizce başka sürüm seçmez, hata verir.
- `import numpy_financial as npf` satırından sonra kütüphane `npf` kısa adıyla kullanılır.
- "Note: you may need to restart the kernel" uyarısı çıkabilir; bu defter için yeniden başlatma gerekmez.

In [ ]:
%pip install -q "numpy-financial==1.1.0"
import numpy_financial as npf
print("numpy-financial:", npf.__version__)

## Bu haftanın sayıları

Veri dosyası yoktur; defter internetten veri okumaz. Girdiler Excel dosyasındaki mavi hücrelerle aynıdır.

| Ne | Değer | Excel'de |
|---|---|---|
| Birikim | her ay 1.000 TL, aylık %3, 12 ay | Girdiler!B5:B7 |
| Tersine hesap | 3 yıl sonra 27.440 TL, yıllık %40 | Girdiler!B10:B12 |
| Kredi (H2 ders notunun 10. bölümündeki örnek) | 100.000 TL, aylık %3, 12 ay; tahsis ücreti 500 TL | Girdiler!B15:B17, Finans, Ileri |
| Proje | bugün -10.000 TL; sonra 3.000, 4.200, 6.800 TL; istenen getiri %10 | Girdiler!B20:B25 |
| TCMB metaverisi örneği | 10.000 TL, 24 ay, ayda 480 TL | Karar!B6:B8 |
| TCMB ihtiyaç kredisi faizi | %63,0 (4 Eylül 2026 haftası) | Karar!B9 |
| İlan kredisi | 100.000 TL, aylık %4, 12 ay | Karar!B48:B50 |
| Üç teklif | A %3,99, 12 ay, 500 TL; B %4,05, 12 ay, ücretsiz; C %3,89, 24 ay, 500 TL | Uygulama!B13:D15 |

%63 dışındaki bütün kredi, proje ve teklif sayıları varsayımsaldır (öğretim için seçilmiştir, piyasa verisi değildir). Tahsis ücretleri yasal sınırdadır: anaparanın binde beşi (TCMB Tebliği 2020/7, md. 10/1).

## Örnek 1: düzenli birikim (Excel: Hesap!B10)

Soru: Her ayın sonunda 1.000 TL yatırılır, aylık faiz %3'tür. 12 ay sonunda hesapta ne birikir?

| Excel | Python |
|---|---|
| `=FV(B5,B6,-B7)` | `npf.fv(aylik_oran, donem, -aylik_katki, 0)` |

Bir fark vardır: `npf.fv` içinde dördüncü argüman (bugünkü değer, `pv`) zorunludur; Excel'de boş bırakılabilir. Başlangıçta hesap boş olduğu için 0 yazılır.

**Tahmin sorusu:** 12 × 1.000 = 12.000 TL yatırılır. Sonuç 12.000'den büyük mü, küçük mü? Neden?

In [ ]:
# [G] Girdiler: Excel Girdiler!B5, B6, B7
aylik_oran = 0.03        # Excel'de %3
donem = 12               # ay
aylik_katki = 1000       # TL, her ayın sonunda

# [D] Oran aylık, dönem aylık, katkı aylık: birimler uyumlu.
# [İ] Yatırılan para cepten çıkar: -aylik_katki (Excel'de -B7)

# [H] Excel: =FV(B5,B6,-B7)
birikim = npf.fv(aylik_oran, donem, -aylik_katki, 0)
print("12 ay sonunda birikim:", birikim)

# [S] Sağlama: Excel Hesap!B10 = 14.192,03
excel_birikim = 14192.03
assert abs(birikim - excel_birikim) < 0.01
print("Sağlama tamam: Python ve Excel aynı sonucu verdi.")

**İnceleme**
- Python sonucu çok ondalıklı yazar. Excel iki ondalık gösterir, çünkü hücreye TL biçimi verilmiştir. Değer aynıdır.
- `abs(a - b) < 0.01`: iki sonucun farkı 1 kuruştan küçük mü? `abs` farkın işaretini atar (Excel'deki ABS). `==` kullanılmaz, çünkü bilgisayar ondalık sayıları yaklaşık tutar.

### İşaret denemesi (Excel: Hesap!B11)

**Tahmin sorusu:** Katkı eksi yerine artı verilirse sonuç ne olur?

In [ ]:
# [İ] İşaret denemesi. Excel: =FV(B5,B6,B7), yani katkı artı
ters_isaret = npf.fv(aylik_oran, donem, aylik_katki, 0)
print("Katkı artı verilince:", ters_isaret)

# [S] Excel Hesap!B11 = -14.192,03
assert abs(ters_isaret - (-14192.03)) < 0.01

### DEĞİŞTİR

1. `yeni_katki` sayısı değiştirilir (örneğin 2000).
2. Excel'de Girdiler!B7'ye aynı sayı yazılır. Hesap!B10'da görülen sonuç `excel_yeni` satırına yazılır. Python'da binlik ayırıcı kullanılmaz, ondalık için nokta kullanılır: `12345.67`.
3. Hücre çalıştırılır. `assert` hata vermezse Python ve Excel yine aynı sonucu bulmuştur.
4. Denemeden sonra Excel'de Girdiler!B7 1000'e geri alınır.

**Tahmin sorusu:** Katkı iki katına çıkarsa birikim de iki katına çıkar mı?

In [ ]:
# DEĞİŞTİR: yeni_katki ve excel_yeni satırları değiştirilir, sonra hücre çalıştırılır.
yeni_katki = 1000
excel_yeni = 14192.03          # Excel Hesap!B10'da görülen sayı

yeni_birikim = npf.fv(aylik_oran, donem, -yeni_katki, 0)
print("Python:", round(yeni_birikim, 2), "  Excel:", excel_yeni)
assert abs(yeni_birikim - excel_yeni) < 0.01, "Python ile Excel farklı: iki taraftaki girdiler kontrol edilmeli."

## Örnek 1b: tersine hesap (Excel: Hesap!B34)

Önceki oturumlarda 10.000 TL'nin yıllık %40 ile 3 yılda 27.440 TL olduğu bulundu. Bu kez soru tersinedir: 3 yıl sonra 27.440 TL'ye sahip olmak için bugün ne yatırılmalıdır?

| Excel | Python |
|---|---|
| `=PV(B32,B33,0,B31)` | `npf.pv(yillik_oran, yil, 0, hedef)` |

**Tahmin sorusu:** Sonucun işareti ne olur?

In [ ]:
# [G] Excel Girdiler!B10, B11, B12
hedef = 27440         # TL, 3 yıl sonra ele geçecek tutar
yillik_oran = 0.40    # Excel'de %40
yil = 3

# [H] Excel: =PV(B32,B33,0,B31). Üçüncü argüman 0: arada düzenli ödeme yok.
bugun = npf.pv(yillik_oran, yil, 0, hedef)
print("Bugün yatırılacak:", bugun)
# Sonda ...0000002 gibi küçük bir artık görülebilir: bilgisayar ondalık sayıları yaklaşık tutar (6. oturum).

# [S] Excel Hesap!B34 = -10.000
assert abs(bugun - (-10000)) < 0.01

## Örnek 2: kredi taksiti (Excel: Hesap!B43 ve Finans!B22)

Çalışılmış örneğin kredisi: 100.000 TL, aylık %3, 12 ay. Aylık taksit ne kadardır?

| Excel | Python |
|---|---|
| `=-PMT(B40,B41,B39)` | `-npf.pmt(kredi_oran, vade, kredi)` |

İki tarafta da fonksiyon taksiti eksi verir, çünkü taksit borçludan çıkan paradır. Başına eksi konarak taksit artı gösterilir.

**Tahmin sorusu:** 12 taksitin toplamı 120.000 TL'den fazla mı?

In [ ]:
# [G] Excel Girdiler!B15, B16, B17
kredi = 100000        # TL, bugün borçluya gelen para
kredi_oran = 0.03     # aylık
vade = 12             # ay

# [İ] npf.pmt taksiti eksi verir; başına eksi konur (Excel'de de öyle).
# [H] Excel: =-PMT(B40,B41,B39)
taksit = -npf.pmt(kredi_oran, vade, kredi)
print("Aylık taksit:", taksit)

# [S] Excel Hesap!B43 = 10.046,21
assert abs(taksit - 10046.21) < 0.01

# [H] Toplam ödeme. Excel Hesap!B64
toplam_odeme = taksit * vade
print("Toplam ödeme:", toplam_odeme)
assert abs(toplam_odeme - 120554.50) < 0.01

### YAZ: toplam faiz (Excel: Hesap!B65 ve Finans!B24)

Aşağıdaki hücrede `None` yerine toplam faizi veren formül yazılır. Excel'deki karşılığı `=B64-B39` (toplam ödeme - kredi tutarı). Yukarıdaki `toplam_odeme` ve `kredi` değişkenleri kullanılır; sayı yazılmaz.

In [ ]:
# YAZ: None yerine formül yazılır.
toplam_faiz = None

In [ ]:
# [S] Kontrol: Excel Hesap!B65 = 20.554,50
if toplam_faiz is None:
    print("Henüz yazılmadı: yukarıdaki hücrede None yerine formül yazılır.")
else:
    assert abs(toplam_faiz - 20554.50) < 0.01, "Excel ile farklı: formül kontrol edilmeli."
    print("Sağlama tamam. Toplam faiz:", round(toplam_faiz, 2))
    print("Yanlış tahmin kredi x oran x vade:", kredi * kredi_oran * vade)

### Amortisman tablosu: hazır bir döngü (Excel: Hesap!A47:J59)

Excel'de amortisman tablosu, formül 12 satır aşağı sürüklenerek kurulur. Python'da aynı işi bir **döngü** yapar: aynı adımları her ay tekrarlar.
1. faiz = kalan borç × oran
2. anapara = taksit - faiz
3. yeni kalan = kalan - anapara
4. son ayda kalan sıfır mı?

**Döngü (`for`) 6. oturumda ayrıntılı işlenir.** Bu oturumda hücre yalnız çalıştırılır ve satırlar Excel tablosuyla karşılaştırılır. İçeriden (girintili) başlayan satırlar her ay tekrarlanır.

**Tahmin sorusu:** Faiz payı aydan aya artar mı, azalır mı?

In [ ]:
# Excel Hesap!A47:J59 tablosunun Python karşılığı. Döngü (for) 6. oturumda işlenir.
kalan = kredi                                  # Dönem 0: borç = kredi tutarı (Excel F47)
for ay in range(1, vade + 1):                  # ay = 1, 2, ..., 12 (Excel A48:A59)
    baslangic = kalan                          # Excel B sütunu: =F47
    faiz = baslangic * kredi_oran              # Excel C sütunu: =B48*$B$40
    anapara = taksit - faiz                    # Excel D sütunu: =E48-C48
    kalan = baslangic - anapara                # Excel F sütunu: =B48-D48
    # [S] Excel G ve I sütunları: =-IPMT($B$40,A48,$B$41,$B$39) ve fark
    faiz_kontrol = -npf.ipmt(kredi_oran, ay, vade, kredi)
    assert abs(faiz - faiz_kontrol) < 0.01
    print(f"{ay:>2}. ay  başlangıç {baslangic:>11,.2f}  faiz {faiz:>9,.2f}  anapara {anapara:>9,.2f}  kalan {kalan:>11,.2f}")

# [S] Son ayda kalan borç 0 olmalı (Excel Hesap!B62)
assert abs(kalan) < 0.01
print("Son ayda kalan borç sıfır: tablo doğru.")

**İnceleme**
- Python sayıları İngilizce biçimde yazar: binlik ayırıcı virgül, ondalık ayırıcı nokta. `92,953.79`, Türkçe yazımla 92.953,79'dur.
- `print(f"...")` satırı yalnız düzgün hizalı yazdırmak içindir (iki ondalık, binlik ayırıcı). Ayrıntısı 6. oturumdadır.
- Her ay `faiz` ile `faiz_kontrol` karşılaştırılır. Bu, Excel'deki "Fark (faiz)" sütununun karşılığıdır.

### DEĞİŞTİR

Yukarıdaki "Örnek 2" hücresinde `vade = 12` satırı `vade = 24` yapılır. Önce o hücre, sonra döngü hücresi çalıştırılır.

**Tahmin sorusu:** Hangi hücre hata verir, neden?

Denemeden sonra `vade = 12` geri yazılır ve iki hücre yeniden çalıştırılır.

## Örnek 3: proje değerlendirme, NPV ve IRR (Excel: Hesap!B78, B79, B85)

Bir projeye bugün 10.000 TL yatırılır. 1., 2. ve 3. yılın sonunda 3.000, 4.200 ve 6.800 TL geri alınır. İstenen getiri yıllık %10'dur.

Python'da akışlar bir **liste** içinde tutulur: köşeli parantez içinde, virgülle ayrılmış, sıralı değerler. Liste 6. oturumda ayrıntılı işlenir.

Excel'de iki sonuç bulunmuştu:
- Doğru kurulum `=NPV(B69,B73:B75)+B72` → 1.307,29 (Hesap!B78)
- Tuzak kurulum `=NPV(B69,B72:B75)` → 1.188,44 (Hesap!B79)

Aşağıdaki hücre `npf.npv` fonksiyonuna yatırım dahil bütün akışları verir.

**Tahmin sorusu:** `npf.npv(0.10, [-10000, 3000, 4200, 6800])` 1.307,29 mu verir, 1.188,44 mü?

In [ ]:
# [G] Excel Girdiler!B20 ve B22:B25
oran = 0.10                                  # yıllık istenen getiri
akislar = [-10000, 3000, 4200, 6800]         # dönem 0 (bugün), 1, 2, 3

# [H] npf.npv: önce oran, sonra akış listesi (yatırım dahil)
nbd = npf.npv(oran, akislar)
print("npf.npv:", nbd)

**İnceleme: zamanlama farkı (çok önemli)**

| | İlk değeri hangi zamana koyar? | Doğru NPV için |
|---|---|---|
| Excel NPV | 1. dönemin sonu (t = 1) | `=NPV(B69,B73:B75)+B72`: yatırım fonksiyonun dışında kalır, sonra eklenir |
| `npf.npv` | bugün (t = 0) | `npf.npv(oran, akislar)`: yatırım listenin başında durur |

Ad aynı, zamanlama farklı. Excel'deki `=NPV(B69,B72:B75)` ile Python'daki `npf.npv(oran, akislar)` **aynı şey değildir**.

In [ ]:
# [S] Excel doğru kurulum =NPV(B69,B73:B75)+B72 -> Hesap!B78 = 1.307,29
assert abs(nbd - 1307.29) < 0.01
print("Sağlama tamam: npf.npv, Excel'in doğru kurulumuyla aynı.")

# Excel TUZAK kurulum =NPV(B69,B72:B75) -> Hesap!B79 = 1.188,44
# Python karşılığı: listenin başına 0 eklemek ("bugün hiçbir şey yok, yatırım bir dönem sonra").
nbd_tuzak = npf.npv(oran, [0] + akislar)
print("Tuzak kurulumun sonucu:", nbd_tuzak)
assert abs(nbd_tuzak - 1188.44) < 0.01

# İki sonucun oranı tam 1 + oran (Excel Hesap!B82)
print("Doğru / tuzak:", nbd / nbd_tuzak)
assert abs(nbd / nbd_tuzak - (1 + oran)) < 0.0001

### İç verim oranı, IRR (Excel: Hesap!B85)

IRR, NPV'yi sıfır yapan orandır. Yatırım dahil bütün akışlar verilir; zamanlama tuzağı yoktur: Excel'deki `=IRR(B72:B75)` ile `npf.irr(akislar)` aynı listeyi alır.

**Tahmin sorusu:** IRR %10'dan büyük mü, küçük mü? (İpucu: %10'da NPV artıydı.)

In [ ]:
# [H] Excel: =IRR(B72:B75) -> Hesap!B85 = %16,34
ivo = npf.irr(akislar)
print("IRR:", ivo)
# Excel %16,34 gösterir; bu yüzden tolerans 0.0001 (yüzde 0,01).
assert abs(ivo - 0.1634) < 0.0001

# [S] IRR'da NPV sıfır olmalı (Excel Hesap!B86)
print("IRR ile hesaplanan NPV:", npf.npv(ivo, akislar))
# e-11 ile biten bir sonuç 0,00000000001 gibi çok küçük bir sayıdır: sıfır kabul edilir.
assert abs(npf.npv(ivo, akislar)) < 0.01

# [Y] Excel: =IF(B78>0,"Kabul","Ret") ve =IF(B85>B69,"Kabul","Ret") -> Hesap!B88, B89
karar = "Kabul" if nbd > 0 else "Ret"
karar_ivo = "Kabul" if ivo > oran else "Ret"
print("NPV kuralı:", karar, "  IRR kuralı:", karar_ivo)
assert karar == "Kabul" and karar_ivo == "Kabul"

### DEĞİŞTİR

1. `yeni_oran` satırı `0.20` yapılır.
2. Excel'de Girdiler!B20 %20 yapılır. Hesap!B78'deki NPV `excel_yeni_nbd` satırına yazılır (eksiyse başına `-` konur).
3. Hücre çalıştırılır. Denemeden sonra Excel'de Girdiler!B20 %10'a geri alınır.

**Tahmin sorusu:** %20'de karar değişir mi? Neden?

In [ ]:
# DEĞİŞTİR: yeni_oran ve excel_yeni_nbd satırları değiştirilir, sonra hücre çalıştırılır.
yeni_oran = 0.10
excel_yeni_nbd = 1307.29      # Excel Hesap!B78'de görülen sayı

yeni_nbd = npf.npv(yeni_oran, akislar)
print("Python NPV:", round(yeni_nbd, 2), "  Excel NPV:", excel_yeni_nbd)
print("Karar:", "Kabul" if yeni_nbd > 0 else "Ret")

# [S] Python ile Excel aynı mı?
assert abs(yeni_nbd - excel_yeni_nbd) < 0.01, "Python ile Excel farklı: iki taraftaki oran kontrol edilmeli."
# [S] Oran IRR'dan küçükse NPV artı, büyükse eksi olmalı.
assert (yeni_nbd > 0) == (yeni_oran < ivo)

## Örnek 4: tahsis ücreti ve kredinin maliyeti (Excel: Finans!B22-B35, Finans!B93-B98, Ileri!B21)

H2 ders notundaki kredi (100.000 TL, aylık %3, 12 ay): aylık %3 yılda yüzde kaç eder, 500 TL tahsis ücreti maliyeti ne kadar artırır? Vakadaki üç teklifin maliyeti aynı yolla bulunur (G2, G3).

| Ne | Excel | Python |
|---|---|---|
| Yıllık nominal | `=B31*B32` | `kredi_oran * 12` |
| Yıllık etkin (bileşik) | `=(1+B31)^B32-1` ya da `=EFFECT(B33,B32)` | `(1 + kredi_oran) ** 12 - 1` |
| Ücretli kredinin aylık maliyeti | `=RATE(B17,-B19,B20)` | `npf.rate(vade, -taksit, ele_gecen, 0)` |
| Aynı oran, akış tablosuyla | `=IRR(B80:B92)` | `npf.irr([ele_gecen] + [-taksit] * vade)` |

`npf.rate`'te dördüncü argüman (gelecekteki değer, `fv`) zorunludur; kredi sonunda borç sıfır olduğu için 0 yazılır.

**Tahmin sorusu:** Ücret taksiti değiştirmez. Aylık maliyet yine %3 mü?

In [ ]:
# [H] Aylık oranın yıllık karşılığı. Excel Finans!B33 ve B34
nominal = kredi_oran * 12
etkin = (1 + kredi_oran) ** 12 - 1
print("Yıllık nominal:", nominal, "  yıllık etkin:", round(etkin, 6))
assert abs(nominal - 0.36) < 0.0001        # Excel %36,00
assert abs(etkin - 0.4258) < 0.0001        # Excel %42,58

# [G] Tahsis ücreti. Excel Ileri!B18 ve Finans!B74
ucret = 500
ele_gecen = kredi - ucret                   # Excel Ileri!B20: 99.500
# [İ] Ele geçen para artı, taksit eksi.
# [H] Excel: =RATE(B17,-B19,B20) -> Ileri!B21
aylik_maliyet = npf.rate(vade, -taksit, ele_gecen, 0)
# [S] Aynı oran, akış tablosu ve IRR ile. Excel Finans!B93 ve fark Finans!B94
aylik_irr = npf.irr([ele_gecen] + [-taksit] * vade)
print("Aylık maliyet, RATE yolu:", round(aylik_maliyet, 6), "  IRR yolu:", round(aylik_irr, 6))
assert abs(aylik_maliyet - 0.0308) < 0.0001      # Excel %3,08
assert abs(aylik_maliyet - aylik_irr) < 1e-9

### YAZ: ücretli kredinin yıllık maliyeti (Excel: Finans!B95)

Aşağıdaki hücrede `None` yerine aylık maliyetin yıllık karşılığını veren formül yazılır. Excel'deki karşılığı `=(1+B93)^B75-1`. Yukarıdaki `etkin` satırındaki kalıp `aylik_maliyet` ile kullanılır; sayı olarak yalnız 1 ve 12 yazılır.

In [ ]:
# YAZ: None yerine formül yazılır.
yillik_maliyet = None

In [ ]:
# [S] Kontrol: Excel Finans!B95 = %43,98; ücretin eklediği Finans!B98 = 1,40 puan
if yillik_maliyet is None:
    print("Henüz yazılmadı: yukarıdaki hücrede None yerine formül yazılır.")
else:
    assert abs(yillik_maliyet - 0.4398) < 0.0001, "Excel ile farklı: formül kontrol edilmeli."
    puan = (yillik_maliyet - etkin) * 100
    print("Vergiler hariç yıllık maliyet:", round(yillik_maliyet, 6))
    print("Ücretin eklediği (yüzde puan):", round(puan, 2))
    assert abs(puan - 1.40) < 0.01

### [Y] Yorum

Örnek 4'ün sonucu tek cümleyle yazılır. Cümlede taksit, toplam faiz ve yıllık maliyet geçer; iki oranın farkı "yüzde puan" diye okunur.

## Vaka: TCMB'nin %63'ü nasıl hesaplanır? (Excel: Karar!B40-B45)

TCMB'nin kredi faizi istatistiğinde her kredinin **efektif yıllık faiz oranı** hesaplanır: sözleşmedeki bütün ödemelerin bugünkü değerini kredi tutarına eşitleyen yıllık oran. Bu, Örnek 4'teki IRR'ın yıllık bileşik karşılığıdır. Metaverideki örnek: 10.000 TL kredi, 24 ay boyunca ayda 480 TL, sonuç %14,90.

Kaynak: TCMB, Kredi Faiz Oranları (Haftalık Akım, Ağırlıklı Ortalama) metaverisi, https://www.tcmb.gov.tr/wps/wcm/connect/33e09fa9-51fb-412f-b38d-0b7cdbaea493/Metaveri_Kredi_Ag%C4%B1rl%C4%B1kl%C4%B1_T%C3%BCrkce.pdf?MOD=AJPERES (erişim 11.10.2026).

| Ne | Excel | Python |
|---|---|---|
| Aylık IRR | `=IRR(C14:C38)` | `npf.irr(akis)` |
| Yıllık bileşik karşılığı | `=(1+B40)^B10-1` | `(1 + aylik) ** 12 - 1` |
| Yıllık orandan aylık orana | `=(1+B9)^(1/B10)-1` | `(1 + yillik) ** (1 / 12) - 1` |

**Tahmin sorusu:** Yıllık %63 aylık yüzde kaçtır? 63 / 12 = %5,25 mi?

In [ ]:
# [G] TCMB metaverisindeki örnek. Excel Karar!B6:B8
tutar_tcmb, odeme_tcmb, vade_tcmb = 10000, 480, 24
# [İ] Ay 0'da kredi borçluya gelir (artı), ay 1-24'te ödeme çıkar (eksi). Excel Karar!C14:C38
akis_tcmb = [tutar_tcmb] + [-odeme_tcmb] * vade_tcmb
# [H] Excel Karar!B40 =IRR(C14:C38)
aylik_tcmb = npf.irr(akis_tcmb)
# [H] Excel Karar!B41 =(1+B40)^B10-1
yillik_tcmb = (1 + aylik_tcmb) ** 12 - 1
print(f"Aylık IRR: {aylik_tcmb:.4%}   yıllık bileşik: {yillik_tcmb:.2%}")
# [S] TCMB metaverisindeki sonuç %14,90
assert abs(yillik_tcmb - 0.1490) < 0.0001

# [G] TCMB ihtiyaç kredisi faizi, 4 Eylül 2026 haftası (PPK özeti, 17.09.2026). Excel Karar!B9
tcmb_ihtiyac = 0.63
aylik_karsilik = (1 + tcmb_ihtiyac) ** (1 / 12) - 1    # Excel Karar!B44
yanlis_okuma = tcmb_ihtiyac / 12                       # Excel Karar!B45
print(f"%63'ün aylık bileşik karşılığı: {aylik_karsilik:.2%}   63 / 12: {yanlis_okuma:.2%}")
assert abs(aylik_karsilik - 0.041555) < 0.000001

---
# Uygulama G3-G4 (Colab)

G1, G2 ve G4'ün Excel kısmı Excel'in **Uygulama** sayfasında yapılır. Bu bölümde G3 (12 dk) ve G4'ün Python kısmı yer alır. Teklifler varsayımsaldır; kredi tavsiyesi değildir.

## G3 (12 dk): taksit, amortisman ve maliyet fonksiyonları

**G3-a.** Üç fonksiyon yazılır:

| Fonksiyon | Ne döndürür | Excel karşılığı |
|---|---|---|
| `taksit(P, r, n)` | Eşit taksit. Kapalı formülle yazılır: P × r / (1 - (1 + r)^(-n)); `npf` kullanılmaz | `=-PMT(r,n,P)` |
| `amortisman(P, r, n)` | Her ay için `(ay, baslangic, faiz, anapara, kalan)` demetlerinden oluşan liste | Uygulama sayfasındaki tablo |
| `maliyet(P, r, n, ucret)` | `(aylik, yillik)`: ele geçen tutar ve taksitlerden oluşan akışın IRR'ı ve yıllık bileşik karşılığı | `=IRR(...)` ve `=(1+x)^12-1` |

**G3-b.** `teklifler` sözlüğündeki üç teklif bir döngüyle hesaplanır; `sonuclar` sözlüğüne her teklif için `(taksit, toplam_faiz, yillik_maliyet)` yazılır ve teklifler yıllık maliyete göre küçükten büyüğe yazdırılır.

In [ ]:
# G3-a: üç fonksiyon. Her fonksiyonda raise satırı silinir ve yerine return içeren kod yazılır.

def taksit(P, r, n):
    """P TL, aylık oran r, n ay: eşit taksit (kapalı formül)."""
    raise NotImplementedError("taksit(P, r, n) yazılmadı")

def amortisman(P, r, n):
    """Her ay için (ay, baslangic, faiz, anapara, kalan) demetlerinin listesi."""
    raise NotImplementedError("amortisman(P, r, n) yazılmadı")

def maliyet(P, r, n, ucret):
    """Tahsis ücretli kredinin (aylık maliyet, yıllık bileşik maliyet) çifti."""
    raise NotImplementedError("maliyet(P, r, n, ucret) yazılmadı")

In [ ]:
# G3-a kontrol: fonksiyonların kendi içinde ve numpy-financial ile tutarlılığı.
# Sonuçlar ayrıca Excel'in Uygulama sayfasındaki değerlerle (E13:G15, C83:C85) karşılaştırılır.
try:
    for oran, vade, ucret in [(0.0399, 12, 500), (0.0389, 24, 500)]:
        t = taksit(100000, oran, vade)
        assert abs(t + npf.pmt(oran, vade, 100000)) < 1e-6                    # PMT ile aynı taksit
        tablo = amortisman(100000, oran, vade)
        assert len(tablo) == vade and abs(tablo[-1][4]) < 1e-6                # son ayda kalan borç 0
        assert abs(sum(s[2] for s in tablo) - (t * vade - 100000)) < 1e-6     # faizler toplamı = toplam faiz
        aylik, yillik = maliyet(100000, oran, vade, ucret)
        akis = [100000 - ucret] + [-t] * vade
        assert abs(npf.npv(aylik, akis)) < 1e-6                               # IRR'da NPV sıfır
        assert abs(yillik - ((1 + aylik) ** 12 - 1)) < 1e-12
    print("G3-a tamam: üç fonksiyon tutarlı. Sayılar Excel Uygulama sayfasıyla karşılaştırılır.")
except NotImplementedError as hata:
    print("Fonksiyonlar henüz yazılmamıştır:", hata)


In [ ]:
# G3-b: üç teklif. Excel Uygulama!B13:D15
teklifler = {"A": (0.0399, 12, 500), "B": (0.0405, 12, 0), "C": (0.0389, 24, 500)}
tutar = 100000

# sonuclar sözlüğü döngüyle kurulur: anahtar teklif harfi, değer (taksit, toplam_faiz, yillik_maliyet).
# Satırların başındaki # işareti kaldırılıp satırlar tamamlanır.
sonuclar = {}
# for ad, (oran, vade, ucret) in teklifler.items():
#     t = ...
#     sonuclar[ad] = (t, ..., ...)

In [ ]:
# G3-b kontrol ve sıralama: Excel Uygulama!F83:F85 (RANK.EQ, 1 = en düşük maliyet)
if not sonuclar:
    print("G3-b henüz yazılmamıştır.")
else:
    sirali = sorted(sonuclar, key=lambda ad: sonuclar[ad][2])
    print("Teklif   taksit (TL)   toplam faiz (TL)   yıllık maliyet")
    for ad in sirali:
        t, f, y = sonuclar[ad]
        print(f"{ad:>6}   {t:11,.2f}   {f:16,.2f}   {y:14.2%}")
    for ad, (oran, vade, ucret) in teklifler.items():
        assert abs(sonuclar[ad][2] - maliyet(tutar, oran, vade, ucret)[1]) < 1e-12
    assert len(sirali) == 3
    print("G3-b tamam. Sıralama Excel'deki RANK.EQ sütunuyla karşılaştırılır.")


**G3-c** (hücreye çift tıklanıp iki cümle yazılır): Akdi oran, toplam faiz ve yıllık maliyet üç teklifi aynı sırayla mı dizer? Farkın nedeni nedir?

...

## G4 (Python kısmı, açık uçlu): tutar × vade ızgarası ve 7.500 TL sınırı

Excel'de Uygulama!A102:F107'deki ızgara Data Table ile kurulur. Python'da aynı ızgara iç içe iki döngüyle kurulur.

**G4-a.** `izgara` sözlüğü kurulur: anahtar `(tutar, vade)`, değer taksit (aylık %4). Taksiti 7.500 TL'yi geçmeyen hücre sayısı `sigan` değişkenine yazılır ve Excel'deki Uygulama!B116 ile karşılaştırılır.

**G4-b.** Excel ızgarası yalnız beş vadeye bakar. Python'da her tutar için taksiti 7.500 TL'nin altına indiren **en kısa vade** 1-120 ay taranarak bulunur ve o vadedeki toplam faiz yazdırılır.

**G4-c.** Üç cümlelik yorum: ne görülmektedir, neden böyledir, ölçünün sınırı nedir?

In [ ]:
# G4-a. Excel Uygulama!B96, B99, A103:A107, B102:F102
oran_g4 = 0.04
sinir = 7500
tutarlar = [50000, 75000, 100000, 125000, 150000]
vadeler = [6, 12, 18, 24, 36]

# Satırların başındaki # işareti kaldırılıp satırlar tamamlanır.
izgara = {}
sigan = None
# for P in tutarlar:
#     for n in vadeler:
#         izgara[(P, n)] = ...
# sigan = ...

In [ ]:
# G4-a kontrol: Excel Uygulama!C105 = 10.655,22 ve B116
if sigan is None:
    print("G4-a henüz yazılmamıştır.")
else:
    print("Tutar / Vade " + "".join(f"{n:>11}" for n in vadeler))
    for P in tutarlar:
        print(f"{P:>12,}" + "".join(f"{izgara[(P, n)]:11,.0f}" for n in vadeler))
    assert abs(izgara[(100000, 12)] - 10655.22) < 0.01
    print("7.500 TL'ye sığan hücre:", sigan)

In [ ]:
# G4-b: her tutar için taksiti sınırın altına indiren en kısa vade (1-120 ay).
# None yerine kod yazılır. İpucu: next(n for n in range(1, 121) if ...)
en_kisa = {P: None for P in tutarlar}

**G4-c yorumu** (üç cümle):

...

## Hızlı şerit (notsuz)

Bu bölüm Python bilen ya da G3-G4'ü bitiren öğrenci içindir; nota etkisi yoktur.

**H-1. Tarihli akış.** TCMB metaverisindeki formül iskontoyu ay sayısıyla değil gün sayısıyla yapar: (1 + EYFO)^(-t/365). Excel'de bu hesap `XIRR` ile yapılır: `=XIRR(C14:C38,B14:B38)` (Karar!B42). `numpy-financial`'da karşılığı yoktur; oran ikiye bölme yöntemiyle bulunur: bugünkü değer farkı sıfırın iki yanında kalacak biçimde aralık her adımda yarıya indirilir.

**H-2. %63'e denk taksit.** 100.000 TL, 12 ay vadeli bir kredide yıllık bileşik maliyetin tam %63 olması için aylık taksit kaç TL olmalıdır? (Tahsis ücreti yok.)

In [ ]:
# H-1: ödeme tarihleri 01.02.2012 ... 01.01.2014 (Excel Karar!B15:B38)
import datetime as dt
bugun = dt.date(2012, 1, 1)
tarihler = [dt.date(2012 + (m - 1) // 12, (m - 1) % 12 + 1, 1) for m in range(2, 26)]
gunler = [(t - bugun).days for t in tarihler]

def bd_farki(e):
    """Kredi tutarı - ödemelerin (1 + e)^(-gün/365) ile iskonto edilmiş toplamı."""
    return tutar_tcmb - sum(odeme_tcmb * (1 + e) ** (-g / 365) for g in gunler)

# YAZ: ikiye bölme. alt ve ust arasında 100 adım; her adımda orta nokta denenir.
eyfo_gunluk = None

In [ ]:
# H-1 kontrol: Excel Karar!B42 ve metaverideki %14,90
if eyfo_gunluk is None:
    print("H-1 henüz yazılmamıştır.")
else:
    print(f"Gün/365 ile: {eyfo_gunluk:.4%}   ay ile: {yillik_tcmb:.4%}")
    assert abs(eyfo_gunluk - 0.148966) < 0.000001

# H-2: %63'e denk taksit. YAZ: None yerine tek satır (G3'teki taksit fonksiyonu ve aylik_karsilik kullanılabilir).
taksit_63 = None

In [ ]:
# H-2 kontrol
if taksit_63 is None:
    print("H-2 henüz yazılmamıştır.")
else:
    print(f"%63'e denk taksit: {taksit_63:,.2f} TL")
    assert round(taksit_63, 2) == 10751.59

## Özet: Excel ve Python

Python sütunundaki argüman adları `numpy-financial`'ın kendi adlarıdır: `rate` dönem oranı, `nper` dönem sayısı, `pmt` dönem ödemesi, `pv` bugünkü değer, `fv` gelecekteki değer, `per` kaçıncı dönem, `values` akış listesi.

| Excel | Python | Not |
|---|---|---|
| FV | `npf.fv(rate, nper, pmt, pv)` | npf'de `pv` zorunlu, Excel'de isteğe bağlı |
| PV | `npf.pv(rate, nper, pmt, fv)` | sıra ve işaret aynı |
| PMT | `npf.pmt(rate, nper, pv)` | ikisi de taksiti eksi verir |
| IPMT | `npf.ipmt(rate, per, nper, pv)` | `per` 1'den başlar |
| PPMT | `npf.ppmt(rate, per, nper, pv)` | faiz + anapara = taksit |
| NPV | `npf.npv(rate, values)` | **Zamanlama farklı:** Excel ilk değeri t = 1, npf t = 0 sayar |
| IRR | `npf.irr(values)` | aynı liste verilir |
| RATE | `npf.rate(nper, pmt, pv, fv)` | npf'de `fv` zorunlu |
| XIRR | yok; ikiye bölme ile | gün / 365 ile iskonto |
| Goal Seek | kapalı çözüm (`npf.pv`) ya da tarama | tek girdi değişir |
| Data Table | iç içe `for` döngüsü | en çok iki girdi |
| IF, AND, OR | `if`, `and`, `or` | Python'da küçük harf |

**Fark olmayan:** işaret kuralı. İki tarafta da borçludan çıkan para eksi, borçluya giren para artıdır.

**Sonraki oturum (20 Ekim):** 5. oturum, Excel III: veriyle çalışma. Vaka dolar ve Bitcoin'in 2025'ten bu yana seyridir; NPV ve IRR örneklerle sürer. 6. oturumda bu defterdeki döngü ve `if` ayrıntılı ele alınır.

**Yapay zekâ kullanımı:** Hata mesajını ve kodun ne yaptığını açıklatmak için kullanılabilir. Kod kopyalanmadan önce her satır açıklanabilmelidir. Deftere ad, e-posta ya da anahtar yazılmaz.